# N07 · KV Cache：为什么 serving 的显存瓶颈常常不是权重？


> 学习方式建议：先读“心智模型”，再运行代码实验，最后做练习题。每道题都附答案解析，不是为了考倒你，而是为了暴露最常见的模棱两可点。  
> 本 notebook 只做概念和可复现小实验；真实工程闭环请回到对应 lab 运行 `make smoke M=...` 并查看 `runs/.../metrics.jsonl`、日志和报告。


## 本节要解决的问题

LLM 推理是自回归生成：每生成一个新 token，都要关注前面所有 token。如果每步都重新计算所有历史 token 的 K/V，代价太高。KV cache 保存历史 token 的 key/value，让 decode 每步只处理新增 token。

但 KV cache 也带来 serving 的核心瓶颈：并发越高、上下文越长、层数越多，显存压力越大。


## 学习地图与版本说明（截至 2026-04-30）

本节从自回归解码的重复计算开始：历史 token 的 K/V 不变，所以缓存它们能避免每步重新计算历史前缀。但缓存也会变成 serving 显存的主要消费者：层数、并发请求数、上下文长度、KV head 数、dtype 都会线性影响 KV cache。很多线上问题不是权重放不下，而是 KV cache 把并发上限压低。

版本上，本教程参考 vLLM PagedAttention 论文、vLLM 官方 blog 与 stable metrics 文档。vLLM 的 PagedAttention 将 KV cache 管理类比操作系统分页，减少动态请求长度造成的碎片和浪费，并支持 prefix sharing 等能力。注意：PagedAttention 优化的是 KV cache 管理，不会消除权重显存，也不会自动修复 tokenizer、调度或网络层瓶颈。

学完本节，你应该能估算“某模型、某上下文、某并发”的 KV cache 大小；能解释 MHA/GQA/MQA 为什么影响 serving 并发；能在 benchmark 中同时看 GPU cache usage、running/waiting requests、TTFT/ITL，而不是只看模型参数量。


## 1. KV cache 保存的是什么？

Transformer attention 中，每层会把 hidden states 投影成 Q/K/V。生成第 t 个 token 时：

- 当前 token 产生新的 Q/K/V。
- Q 需要和历史所有 K 做 attention。
- 输出需要历史所有 V 加权求和。

历史 token 的 K/V 不变，所以可以缓存。KV cache 形状粗略理解为：

```text
[layer, batch_or_requests, seq_len, kv_heads, head_dim, K/V]
```

这就是为什么长上下文和高并发会线性增加 KV cache。


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def kv_cache_gb(layers=32, seq_len=4096, concurrent=8, hidden=4096, num_heads=32, kv_heads=None, bytes_per_elem=2):
    # kv_heads 支持 GQA/MQA。若 None，按 MHA kv_heads=num_heads。
    kv_heads = num_heads if kv_heads is None else kv_heads
    head_dim = hidden // num_heads
    total_bytes = layers * concurrent * seq_len * kv_heads * head_dim * 2 * bytes_per_elem
    return total_bytes / 1024**3

rows = []
for concurrent in [1, 4, 8, 16, 32]:
    for seq in [1024, 4096, 8192, 32768]:
        rows.append({
            "并发": concurrent,
            "上下文长度": seq,
            "MHA_KV_GB": kv_cache_gb(seq_len=seq, concurrent=concurrent, kv_heads=32),
            "GQA_8KV_GB": kv_cache_gb(seq_len=seq, concurrent=concurrent, kv_heads=8),
        })

df = pd.DataFrame(rows)
display(df.round(2).head(12))
df[df["并发"] == 8].plot(x="上下文长度", y=["MHA_KV_GB", "GQA_8KV_GB"], marker="o", title="并发=8 时 KV cache 估算")
plt.ylabel("GB")
plt.show()


## 2. MHA、GQA、MQA 对 KV cache 的影响

- **MHA**：每个 query head 都有对应 K/V head，KV cache 最大。
- **GQA**：多个 query head 共享一组 K/V head，KV cache 变小。
- **MQA**：所有 query heads 共享很少 K/V head，KV cache 更小。

这解释了为什么同样参数量的模型，serving 并发能力可能差很多：KV head 数和上下文长度同样重要。


## 3. PagedAttention 的直觉

vLLM 的 PagedAttention 把 KV cache 类比为操作系统分页：逻辑上连续的 token，不要求物理显存连续。这样可以减少碎片和过度预留，并支持不同请求共享相同 prompt 的 cache block。

工程意义：

- KV cache 不再需要为最大长度预留一整块连续空间。
- 动态请求长度下显存浪费更少。
- 更容易做 prefix sharing / parallel sampling 的共享。

但注意：PagedAttention 管理的是 KV cache 内存，不会让模型权重消失，也不能解决所有延迟问题。


## 3.5 RadixCache vs PagedAttention：策略层 vs 寻址层

很多人把 RadixCache 和 PagedAttention 当作"二选一"，其实它们解决的根本不是同一个问题——在 SGLang / vLLM 现行版本里，两者都默认开启，类比操作系统的"两级索引"：

| 维度 | PagedAttention（寻址层） | RadixCache（策略层） |
|---|---|---|
| 解决的问题 | **显存碎片**：逻辑连续 → 物理离散 | **重复计算**：多请求间共享前缀 KV |
| OS 类比 | **页表**（虚拟地址 → 物理页） | **共享库 / 文件缓存**（多进程共用同一物理页） |
| 关注点 | 怎么**存**：利用不连续的显存块 | 存**什么**：哪些前缀可复用 |
| 关系 | 没有分页，共享会被碎片堵死 | 利用分页提供的灵活性，做极致复用 |

**记忆口诀**：PagedAttention 让 KV 能放下，RadixCache 让 KV 不重算。

参考 [Zhaochen Yang · 从 KV Cache 到 Zero Overhead Scheduling](../github_repo/Awesome-ML-SYS-Tutorial/sglang/scheduler/readme.md)。

## 4. KV cache 与吞吐/延迟的关系

KV cache 既省计算，又吃显存：

- 没有 KV cache：每步重新处理历史，计算爆炸。
- 有 KV cache：decode 每步只算新增 token，但要读取历史 K/V，长上下文时 memory bandwidth 压力大。
- 并发太高：KV cache 占满显存，请求排队或被抢占，TTFT/ITL 变差。

因此 serving 调优必须同时看：`request_prompt_tokens`、`request_generation_tokens`、`kv_cache_usage`、TTFT、ITL、queue time。


In [ ]:
def capacity_estimate(gpu_gb=80, weight_gb=28, reserved_gb=6, per_request_kv_gb=1.2):
    available = max(0, gpu_gb - weight_gb - reserved_gb)
    return int(available // per_request_kv_gb)

for ctx in [2048, 8192, 32768]:
    per_req = kv_cache_gb(seq_len=ctx, concurrent=1, kv_heads=8)
    print({"ctx": ctx, "per_request_kv_gb": round(per_req, 2), "估算最大并发": capacity_estimate(per_request_kv_gb=per_req)})


## 5. 与本课程的连接

- L07 vLLM baseline：观察 TTFT/ITL 和 KV cache usage。
- L08 SGLang：prefix cache / RadixAttention 与 repeated-prefix benchmark。
- L09 PD：prefill 和 decode 拆开后，decode 仍受 KV cache 读取影响。
- Debug ticket：`vllm_memory_pressure_001`、`sglang_high_ttft_001`。


## 6. 企业面试/工程判断痛点题（带答案）

### 题 1：7B bf16 权重约 14GB，为什么 24GB GPU serving 并发仍可能很低？

**答案解析：** 还要留 KV cache、CUDA context、allocator、临时 buffer。长上下文和高并发会让 KV cache 快速吃掉剩余显存。

### 题 2：KV cache 是减少 TTFT 还是 ITL？

**答案解析：** 基础 KV cache 主要让 decode 每步不用重算历史，显著改善后续 token 生成成本；prefix cache 命中时可以减少重复 prefill，从而改善 TTFT。

### 题 3：GQA 为什么有利于 serving？

**答案解析：** GQA 减少 KV heads 数，KV cache 按 kv_heads 线性下降，从而提高长上下文/高并发能力。

### 题 4：PagedAttention 解决的主要问题是什么？

**答案解析：** 主要解决动态长度请求下 KV cache 内存碎片和预留浪费，并支持灵活共享。它不是模型压缩，也不是让 attention 计算复杂度消失。

### 题 5：如果 TTFT 很高但 ITL 正常，优先怀疑什么？

**答案解析：** 优先看 queue time、prefill 长度、prefix cache 是否 miss、tokenization/input processing，而不是 decode 阶段。


## 参考资料

- vLLM PagedAttention blog: https://vllm.ai/blog/vllm
- PagedAttention paper: https://arxiv.org/abs/2309.06180
- vLLM metrics: https://docs.vllm.ai/en/stable/design/metrics/
